# Задача 08. Эффективность промокодов

**Постановка задачи:** нужно оценить, какие промокоды используются чаще и какую выручку они приносят. Для доставленных заказов посчитать:

- количество заказов;
- количество покупателей;
- выручку;
- средний чек;
- среднюю скидку;
- долю повторных покупателей среди пользователей промокода.

Заказы без промокода выделить в отдельную группу `no_promo`.

In [ ]:
import sqlite3
from pathlib import Path

import pandas as pd

DB_PATH = Path('../data/marketplace.sqlite')
conn = sqlite3.connect(DB_PATH)

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:,.2f}'.format)

print(f'База подключена: {DB_PATH.resolve()}')

In [ ]:
query = r"""
WITH order_finance AS (
    SELECT
        o.order_id,
        o.customer_id,
        COALESCE(o.promo_code, 'no_promo') AS promo_code,
        SUM(oi.quantity * oi.unit_price * (1 - oi.discount_pct / 100.0)) AS revenue,
        AVG(oi.discount_pct) AS avg_discount_pct
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    WHERE o.status = 'delivered'
    GROUP BY o.order_id, o.customer_id, COALESCE(o.promo_code, 'no_promo')
), customer_orders AS (
    SELECT
        customer_id,
        COUNT(order_id) AS delivered_orders
    FROM order_finance
    GROUP BY customer_id
)
SELECT
    of.promo_code,
    COUNT(of.order_id) AS orders,
    COUNT(DISTINCT of.customer_id) AS buyers,
    ROUND(SUM(of.revenue), 2) AS revenue,
    ROUND(AVG(of.revenue), 2) AS avg_order_value,
    ROUND(AVG(of.avg_discount_pct), 2) AS avg_discount_pct,
    ROUND(100.0 * COUNT(DISTINCT CASE WHEN co.delivered_orders >= 2 THEN of.customer_id END) / COUNT(DISTINCT of.customer_id), 2) AS repeat_buyers_pct
FROM order_finance of
JOIN customer_orders co ON of.customer_id = co.customer_id
GROUP BY of.promo_code
ORDER BY revenue DESC;
"""

df = pd.read_sql_query(query, conn)
df

**Комментарий стажёра:** промокод полезно смотреть не только по выручке, но и по качеству покупателей: возвращаются ли они после заказа.